# Linear Regression from Scratch

## Problem
Predict house prices from square footage, bedrooms and age, and show that I can fit the model using only matrix operations (no scikit-learn).

## Approach
1. Generate seeded housing data from a known formula, so the learned coefficients can be checked against the truth.
2. Solve the **normal equation** $\theta = (X^TX)^{-1}X^Ty$ after checking that $\det(X^TX) \neq 0$.
3. Fit the same model with **vectorized gradient descent** using the `@` operator.
4. Evaluate on a held-out test set with RMSE and R².

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)  # reproducible
print(np.__version__)

## 1. Generate seeded housing data with a known formula

In [ ]:
n = 500

sqft     = np.random.uniform(600, 3500, n)
bedrooms = np.random.randint(1, 6, n)
age      = np.random.uniform(0, 50, n)

# Known formula: price = 50,000 + 150*sqft + 10,000*bedrooms - 800*age + noise
true_theta = np.array([50_000, 150, 10_000, -800])
X = np.column_stack([np.ones(n), sqft, bedrooms, age])   # first column = bias term
y = X @ true_theta + np.random.normal(0, 15_000, n)

# Seeded 80/20 train-test split
idx = np.random.permutation(n)
split = int(0.8 * n)
train, test = idx[:split], idx[split:]
X_train, X_test = X[train], X[test]
y_train, y_test = y[train], y[test]

print("Train:", X_train.shape, "| Test:", X_test.shape)

## 2. Check the determinant, then solve with inverse and transpose
$\theta = (X^TX)^{-1}X^Ty$

In [ ]:
A = X_train.T @ X_train
det = np.linalg.det(A)
print(f"det(XᵀX) = {det:.3e}  ->", "invertible" if abs(det) > 0 else "singular!")
print(f"condition number = {np.linalg.cond(A):.3e}")

theta_ne = np.linalg.inv(A) @ X_train.T @ y_train

print("\n            bias     sqft  bedrooms      age")
print("true     ", np.round(true_theta, 1))
print("normal eq", np.round(theta_ne, 1))

## 3. Gradient descent using the `@` operator
Features are standardized so one learning rate works for every column.

In [ ]:
mu    = X_train[:, 1:].mean(axis=0)
sigma = X_train[:, 1:].std(axis=0)

def scale(X):
    return np.column_stack([np.ones(len(X)), (X[:, 1:] - mu) / sigma])

Xs_train, Xs_test = scale(X_train), scale(X_test)

m = len(y_train)
theta_s = np.zeros(Xs_train.shape[1])
lr, epochs = 0.1, 1000
loss_history = []

for _ in range(epochs):
    error = Xs_train @ theta_s - y_train
    theta_s -= lr * (Xs_train.T @ error) / m
    loss_history.append((error ** 2).mean())

# Convert back to original units to compare with the normal equation
theta_gd = np.empty_like(theta_s)
theta_gd[1:] = theta_s[1:] / sigma
theta_gd[0]  = theta_s[0] - np.sum(theta_s[1:] * mu / sigma)

print("normal eq", np.round(theta_ne, 1))
print("grad desc", np.round(theta_gd, 1))

plt.plot(loss_history)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.title("Gradient descent convergence")
plt.show()

## 4. RMSE, R² and a predicted-vs-actual plot

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    return 1 - ss_res / ss_tot

pred_ne = X_test @ theta_ne
pred_gd = X_test @ theta_gd

print(f"Normal equation  -> RMSE: {rmse(y_test, pred_ne):,.0f} | R²: {r2(y_test, pred_ne):.4f}")
print(f"Gradient descent -> RMSE: {rmse(y_test, pred_gd):,.0f} | R²: {r2(y_test, pred_gd):.4f}")

plt.scatter(y_test, pred_ne, alpha=0.6)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, "r--", label="Perfect prediction")
plt.xlabel("Actual price"); plt.ylabel("Predicted price")
plt.title("Predicted vs actual (test set)")
plt.legend(); plt.show()

## Results
- Both methods recovered coefficients close to the true formula, and normal equation and gradient descent agree.
- Test RMSE: **14,852** (normal equation), **14,852** (gradient descent)
- Test R²: **0.9878** (normal equation), **0.9878** (gradient descent)
- The data has noise with a standard deviation of 15,000, so an RMSE near that is about the best any model can do here.